# StackOverflow Unified Indexing (Lucene + Forward)

This notebook creates one unified collection named `stackoverflow_all` and indexes the whole dataset once.

It assumes you already have per-split collections (for example: `stackoverflow_train`, `stackoverflow_test`, `stackoverflow_dev1`, `stackoverflow_dev2`, `stackoverflow_tran`) produced by your earlier notebooks.

The notebook will:
1. Validate source split collections.
2. Build `collections/stackoverflow_all/input_data/<split>/...` by copying split folders.
3. Build one Lucene index over all splits.
4. Build forward indexes over all splits.

In [1]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()

SOURCE_COLLECTIONS = [
    'stackoverflow_train',
    'stackoverflow_test',
    'stackoverflow_dev1',
    'stackoverflow_dev2',
    'stackoverflow_tran'
]

TARGET_COLLECTION = 'stackoverflow_all'
INPUT_SUBDIR = 'input_data'

# Lucene index field used by your StackOverflow setup
LUCENE_INDEX_FIELD = 'text'

# Forward index fields
FIELD_DEFS = [
    'text:parsedText',
    'text_unlemm:parsedText',
    'text_raw:textRaw'
]

# If True, remove stackoverflow_all/input_data before copying split folders
REBUILD_INPUT_DATA = False

# If True, pass -clean to create_fwd_index.sh
CLEAN_FWD_INDEX = False

print('REPO_ROOT =', REPO_ROOT)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('COLLECT_ROOT =', COLLECT_ROOT)
print('SOURCE_COLLECTIONS =', SOURCE_COLLECTIONS)
print('TARGET_COLLECTION =', TARGET_COLLECTION)
print('LUCENE_INDEX_FIELD =', LUCENE_INDEX_FIELD)
print('FIELD_DEFS =', FIELD_DEFS)
print('REBUILD_INPUT_DATA =', REBUILD_INPUT_DATA)
print('CLEAN_FWD_INDEX =', CLEAN_FWD_INDEX)

REPO_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
SOURCE_COLLECTIONS = ['stackoverflow_train', 'stackoverflow_test', 'stackoverflow_dev1', 'stackoverflow_dev2', 'stackoverflow_tran']
TARGET_COLLECTION = stackoverflow_all
LUCENE_INDEX_FIELD = text
FIELD_DEFS = ['text:parsedText', 'text_unlemm:parsedText', 'text_raw:textRaw']
REBUILD_INPUT_DATA = False
CLEAN_FWD_INDEX = False


## 1) Validate source split collections

Each source collection is expected to have `input_data/<one split dir>/` that contains:
- `QuestionFields.jsonl`
- `AnswerFields.jsonl`
- `qrels.txt`

In [3]:
source_infos = []

for coll in SOURCE_COLLECTIONS:
    coll_dir = COLLECT_ROOT / coll
    if not coll_dir.exists():
        raise FileNotFoundError(f'Missing collection directory: {coll_dir}')

    input_data_dir = coll_dir / INPUT_SUBDIR
    if not input_data_dir.exists():
        raise FileNotFoundError(f'Missing input_data directory: {input_data_dir}')

    split_subdirs = sorted([p for p in input_data_dir.iterdir() if p.is_dir()])
    if not split_subdirs:
        raise FileNotFoundError(f'No split subdirectory found in {input_data_dir}')

    if len(split_subdirs) != 1:
        raise RuntimeError(
            f'Expected exactly one split subdirectory in {input_data_dir}, found {len(split_subdirs)}: {[p.name for p in split_subdirs]}'
        )

    split_dir = split_subdirs[0]
    split_name = split_dir.name

    required = [
        split_dir / 'QuestionFields.jsonl',
        split_dir / 'AnswerFields.jsonl',
        split_dir / 'qrels.txt'
    ]
    for p in required:
        if not p.exists():
            raise FileNotFoundError(f'Missing required file: {p}')

    source_infos.append((coll, split_name, split_dir))
    print(f'{coll} -> split={split_name} ({split_dir})')

print('All source collections are valid.')

stackoverflow_train -> split=train (/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_train/input_data/train)
stackoverflow_test -> split=test (/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_test/input_data/test)
stackoverflow_dev1 -> split=dev1 (/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_dev1/input_data/dev1)
stackoverflow_dev2 -> split=dev2 (/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_dev2/input_data/dev2)
stackoverflow_tran -> split=tran (/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/input_data/tran)
All source collections are valid.


## 2) Build unified collection layout

This creates `collections/stackoverflow_all/input_data/<split>/...` by copying each split folder from source collections.

In [4]:
target_collect_dir = COLLECT_ROOT / TARGET_COLLECTION
target_input_data_dir = target_collect_dir / INPUT_SUBDIR

target_collect_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_INPUT_DATA and target_input_data_dir.exists():
    print('Removing existing input_data:', target_input_data_dir)
    shutil.rmtree(target_input_data_dir)

target_input_data_dir.mkdir(parents=True, exist_ok=True)

for src_coll, split_name, src_split_dir in source_infos:
    dst_split_dir = target_input_data_dir / split_name

    if dst_split_dir.exists():
        raise FileExistsError(
            f'Target split already exists: {dst_split_dir}. '
            'Set REBUILD_INPUT_DATA=True to recreate input_data from scratch.'
        )

    shutil.copytree(src_split_dir, dst_split_dir)
    print(f'Copied {src_coll}:{split_name} -> {dst_split_dir}')

print('Unified collection input_data is ready:', target_input_data_dir)
print('Splits in target:', sorted([p.name for p in target_input_data_dir.iterdir() if p.is_dir()]))

Copied stackoverflow_train:train -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data/train
Copied stackoverflow_test:test -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data/test
Copied stackoverflow_dev1:dev1 -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data/dev1
Copied stackoverflow_dev2:dev2 -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data/dev2
Copied stackoverflow_tran:tran -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data/tran
Unified collection input_data is ready: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data
Splits in target: ['dev1', 'dev2', '

## 3) Build one Lucene index over all splits

`create_lucene_index.sh` is run once on `stackoverflow_all` and scans all split subdirectories under `input_data`.

In [5]:
env = os.environ.copy()
env['COLLECT_ROOT'] = str(COLLECT_ROOT)

cmd_lucene = [
    'bash', './index/create_lucene_index.sh',
    TARGET_COLLECTION,
    '-input_subdir', INPUT_SUBDIR,
    '-index_field', LUCENE_INDEX_FIELD
]

print('Running:', ' '.join(cmd_lucene))
res = subprocess.run(cmd_lucene, cwd=SCRIPTS_DIR, env=env, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'create_lucene_index.sh failed with code {res.returncode}')

Running: bash ./index/create_lucene_index.sh stackoverflow_all -input_subdir input_data -index_field text
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
Input data directory: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data
Index directory:      /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/lucene_index
Index field name:     text
Exact match param:    
Ignore missing field: 
Checking input sub-directory: dev1
Found indexable data file: dev1/AnswerFields.jsonl
Checking input sub-directory: dev2
Found indexable data file: dev2/AnswerFields.jsonl
Checking input sub-directory: test
Found indexable data file: test/AnswerFields.jsonl
Checking input sub-directory: train
Found indexable data file: train/AnswerFields.jsonl
Checking input sub-directory: tran
Found indexable data file: tran/Answe

## 4) Build forward indexes over all splits

`create_fwd_index.sh` is run once per field definition on `stackoverflow_all`.

In [6]:
for field_def in FIELD_DEFS:
    cmd_fwd = ['bash', './index/create_fwd_index.sh', TARGET_COLLECTION, field_def]
    if CLEAN_FWD_INDEX:
        cmd_fwd.append('-clean')

    print('Running:', ' '.join(cmd_fwd))
    res = subprocess.run(cmd_fwd, cwd=SCRIPTS_DIR, env=env, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'create_fwd_index.sh failed for {field_def} with code {res.returncode}')

Running: bash ./index/create_fwd_index.sh stackoverflow_all text:parsedText
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
Input data directory:      /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/input_data
Forward index directory:   /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/forward_index
Remove old index?:         0
Field definition:          text:parsedText
Index type arguments:        
Checking input sub-directory: dev1
Found indexable data file: dev1/AnswerFields.jsonl
Checking input sub-directory: dev2
Found indexable data file: dev2/AnswerFields.jsonl
Checking input sub-directory: test
Found indexable data file: test/AnswerFields.jsonl
Checking input sub-directory: train
Found indexable data file: train/AnswerFields.jsonl
Checking input sub-directory: tran
Found indexable data file: 

## 5) Verify output

Checks the unified collection structure and index directories.

In [7]:
target_collect_dir = COLLECT_ROOT / TARGET_COLLECTION
target_input_data_dir = target_collect_dir / INPUT_SUBDIR
lucene_dir = target_collect_dir / 'lucene_index'
fwd_dir = target_collect_dir / 'forward_index'

print('Target collection:', target_collect_dir)
print('input_data exists:', target_input_data_dir.exists())
if target_input_data_dir.exists():
    print('input_data splits:', sorted([p.name for p in target_input_data_dir.iterdir() if p.is_dir()]))

print('lucene_index exists:', lucene_dir.exists())
if lucene_dir.exists():
    print('lucene_index sample:', sorted([p.name for p in lucene_dir.iterdir()])[:20])

print('forward_index exists:', fwd_dir.exists())
if fwd_dir.exists():
    print('forward_index sample:', sorted([p.name for p in fwd_dir.iterdir()])[:20])

Target collection: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all
input_data exists: True
input_data splits: ['dev1', 'dev2', 'test', 'train', 'tran']
lucene_index exists: True
lucene_index sample: ['_0.cfe', '_0.cfs', '_0.si', 'segments_1', 'write.lock']
forward_index exists: True
forward_index sample: ['text', 'text.mapdb_dataDict', 'text_raw', 'text_raw.mapdb_dataDict', 'text_unlemm', 'text_unlemm.mapdb_dataDict']


## Notes

- If `stackoverflow_all/input_data` already exists, either set `REBUILD_INPUT_DATA = True` or manually remove old split folders.
- Lucene indexing script always recreates the target Lucene index directory.
- Keep using this unified collection (`stackoverflow_all`) in downstream experiment notebooks.